# Урок 19 · AutoML: «дай машине делать ML»

Сегодня узнаем, как получить сильную модель почти без кода. AutoML сам перебирает разные модели,
честно проверяет их и выдаёт лучшую. Мы сравним: кто победит — модель, которую МЫ соберём руками,
или AutoML за 3 минуты?

> План: 1) руками собираем модель (как раньше) → 2) AutoML на тех же данных → 3) сравниваем → 4) зоопарк Hugging Face Hub.

## Что такое AutoML простыми словами

AutoML — это НЕ новая волшебная модель. Это «робот-помощник», который делает за тебя то, что ты уже умеешь:
обучает много разных моделей (KNN, дерево, лес, ансамбли), честно проверяет каждую и показывает таблицу
от лучшей к худшей. Ты видел это в демо-гонке. Теперь запустим по-настоящему.

Используем **AutoGluon** — самый простой AutoML для таблиц.

## Шаг 0. Данные — знакомые пингвины

Берём Palmer Penguins. Задача: по измерениям определить вид пингвина (`species`). Это классификация.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split

# грузим пингвинов, убираем пропуски и колонку sex (по стандарту курса)
df = sns.load_dataset("penguins").dropna().drop(columns=["sex"])

# делим на train/test с random_state — честное сравнение
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Обучающих:", len(train_df), "| Тестовых:", len(test_df))
print("Виды:", df["species"].unique())
df.head()

## Шаг 1. Сначала — РУКАМИ (как на прошлых уроках)

Соберём модель через sklearn Pipeline сами, чтобы было с чем сравнивать AutoML.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
cat = ["island"]

prep = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

manual = Pipeline([("prep", prep),
                   ("clf", RandomForestClassifier(random_state=42))])

manual.fit(train_df.drop(columns=["species"]), train_df["species"])
manual_acc = manual.score(test_df.drop(columns=["species"]), test_df["species"])
print(f"Наша модель РУКАМИ: точность = {manual_acc:.3f}")

**❓ Вопрос 1.** Сколько строк кода ушло на модель руками? Запомни это число — сравним с AutoML.

## Шаг 2. Теперь — AutoML (AutoGluon)

Устанавливаем и запускаем. Обрати внимание: НЕ нужно вручную выбирать модель, масштабировать, кодировать —
AutoGluon делает всё сам. Мы только говорим: «вот данные, вот что предсказывать».

In [ ]:
!pip install autogluon.tabular -q

In [ ]:
from autogluon.tabular import TabularPredictor

# ВСЯ подготовка модели — в одной строке fit.
# label = что предсказываем. time_limit = сколько секунд дать на перебор.
predictor = TabularPredictor(label="species").fit(
    train_df,
    time_limit=120,          # 2 минуты на перебор моделей
    presets="medium_quality" # баланс скорости и качества
)
print("\nAutoML закончил перебор!")

> При установке некоторые модели (xgboost, torch-сети) могут пропускаться с предупреждением — это нормально, AutoGluon обучит те, что доступны. На точность урока это почти не влияет.

## Шаг 3. Таблица моделей (leaderboard)

Вот сердце AutoML: таблица всех моделей, которые он обучил, от лучшей к худшей.
Именно это ты видел в демо-гонке — только теперь на реальных данных.

In [ ]:
# leaderboard показывает все модели и их точность на тесте
lb = predictor.leaderboard(test_df, silent=True)
lb[["model", "score_test"]]

**❓ Вопрос 2.** Какая модель победила? Это одна из знакомых (лес, extra trees) или ансамбль нескольких?
Сколько всего моделей перебрал AutoML?

## Шаг 4. Сравниваем: руками vs AutoML

In [ ]:
# лучшая точность AutoML — верхняя строка leaderboard
automl_acc = lb["score_test"].max()

print(f"Модель РУКАМИ:  {manual_acc:.3f}")
print(f"AutoML:         {automl_acc:.3f}")
print(f"Разница:        {automl_acc - manual_acc:+.3f}")

if automl_acc >= manual_acc:
    print("\nAutoML не хуже нашей ручной модели — и почти без кода!")
else:
    print("\nНаша ручная модель оказалась не хуже — тоже отличный результат!")

**❓ Вопрос 3.** AutoML сильно обогнал ручную модель? На маленьких чистых данных (как пингвины)
разница часто мала. На больших грязных данных AutoML обычно выигрывает заметнее. Почему, как думаешь?

## Шаг 5. Предсказание на новых данных

Модель готова — используем её так же просто, как обычную.

In [ ]:
# берём одного пингвина из теста (без ответа) и предсказываем вид
one = test_df.drop(columns=["species"]).iloc[[0]]
pred = predictor.predict(one)

print("Предсказанный вид:", pred.values[0])
print("Правильный ответ: ", test_df["species"].iloc[0])

## Шаг 6. Зоопарк готовых моделей — Hugging Face Hub

AutoML строит модель под ТВОИ данные. А если задача типовая (тональность, перевод, распознавание) —
модель уже кто-то обучил и выложил на **Hugging Face Hub**. Это огромный «зоопарк» готовых моделей.
Ты уже брал оттуда модель на уроке 18!

Зайди на huggingface.co/models, найди модель для задачи, которая тебе интересна, и посмотри её карточку.

In [ ]:
# напоминание из урока 18: готовая модель одной строкой
from transformers import pipeline
clf = pipeline("sentiment-analysis")
print(clf("AutoML makes machine learning so much easier!"))

---
## 🎯 Задания

### 🟢 Базовый
Запусти AutoML на пингвинах, найди в leaderboard лучшую модель и её точность. Сравни с ручной моделью. Запиши обе цифры.

### 🟡 Продвинутый
Поменяй задачу на РЕГРЕССИЮ: предскажи `body_mass_g` (вес) вместо вида. Достаточно поменять `label` и убрать `species`? Запусти AutoML и посмотри, какая метрика теперь в leaderboard (не accuracy!).

### ⭐ Со звёздочкой
Возьми свой датасет с Kaggle (табличный, с целевой колонкой), запусти на нём AutoGluon. Что в топе leaderboard? Совпало ли с твоей интуицией?

## Мини-итог

- AutoML — это ... (не магия, а ...)
- leaderboard показывает ...
- Когда лучше AutoML, а когда готовая модель с Hugging Face Hub?

> Ты научился получать сильную модель почти без кода — и понимаешь, ЧТО происходит внутри. Это не магия, это перебор, который ты уже умел делать руками.